In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

protein_path = ROOT / "data/processed/human_proteins_with_string.parquet"
gaf_path = ROOT / "data/raw/go/goa_human.gaf.gz"

protein_df = pd.read_parquet(protein_path)

print("Canonical proteins:", len(protein_df))
print("GOA Human file exists:", gaf_path.exists())

Canonical proteins: 18841
GOA Human file exists: True


In [2]:
# GAF 2.2 column names
GAF_COLUMNS = [
    "DB",
    "DB_Object_ID",
    "DB_Object_Symbol",
    "Qualifier",
    "GO_ID",
    "DB_Reference",
    "Evidence_Code",
    "With_or_From",
    "Aspect",
    "DB_Object_Name",
    "DB_Object_Synonym",
    "DB_Object_Type",
    "Taxon",
    "Date",
    "Assigned_By",
    "Annotation_Extension",
    "Gene_Product_Form_ID",
]

gaf_df = pd.read_csv(
    gaf_path,
    sep="\t",
    comment="!",
    header=None,
    names=GAF_COLUMNS,
    compression="gzip",
    dtype=str,
)

print("Raw GO annotation rows:", f"{len(gaf_df):,}")
print("Columns:", gaf_df.columns.tolist())

gaf_df.head()

Raw GO annotation rows: 906,445
Columns: ['DB', 'DB_Object_ID', 'DB_Object_Symbol', 'Qualifier', 'GO_ID', 'DB_Reference', 'Evidence_Code', 'With_or_From', 'Aspect', 'DB_Object_Name', 'DB_Object_Synonym', 'DB_Object_Type', 'Taxon', 'Date', 'Assigned_By', 'Annotation_Extension', 'Gene_Product_Form_ID']


,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,Evidence_Code,With_or_From,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_By,Annotation_Extension,Gene_Product_Form_ID
0,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0005515,PMID:33961781,IPI,UniProtKB:Q8NFP7,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20260425,IntAct,NaN,UniProtKB:A0A024RBG1
1,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0008486,GO_REF:0000003,IEA,EC:3.6.1.52,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20260428,UniProt,NaN,UniProtKB:A0A024RBG1
2,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0008486,GO_REF:0000024,ISS,UniProtKB:Q9NZJ9,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20240703,UniProt,NaN,UniProtKB:A0A024RBG1
3,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0016462,GO_REF:0000002,IEA,InterPro:IPR047198,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20260428,InterPro,NaN,UniProtKB:A0A024RBG1
4,UniProtKB,A0A024RBG1,NUDT4B,enables,GO:0016787,GO_REF:0000002,IEA,InterPro:IPR020084,F,Diphosphoinositol polyphosphate phosphohydrola...,NUDT4B,protein,taxon:9606,20260428,InterPro,NaN,UniProtKB:A0A024RBG1


In [3]:
# "NOT" annotations function absent hone ko show karte hain;
# unhe positive prediction labels mein include nahi karna chahiye.
is_not_annotation = gaf_df["Qualifier"].fillna("").str.contains(
    r"(^|\|)NOT(\||$)",
    regex=True,
)

positive_gaf_df = gaf_df.loc[~is_not_annotation].copy()

protein_ids = set(protein_df["uniprot_id"])

matched_gaf_df = positive_gaf_df[
    positive_gaf_df["DB_Object_ID"].isin(protein_ids)
].copy()

print("GO rows after removing NOT annotations:", f"{len(positive_gaf_df):,}")
print("GO rows matched to our protein table:", f"{len(matched_gaf_df):,}")
print("Unique matched proteins:", f"{matched_gaf_df['DB_Object_ID'].nunique():,}")

print("\nGO aspect counts:")
print(matched_gaf_df["Aspect"].value_counts())

print("\nMost common evidence codes:")
print(matched_gaf_df["Evidence_Code"].value_counts().head(10))

C:\Users\rajdeep\AppData\Local\Temp\ipykernel_31108\3468309190.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_not_annotation = gaf_df["Qualifier"].fillna("").str.contains(


GO rows after removing NOT annotations: 904,944
GO rows matched to our protein table: 838,361
Unique matched proteins: 18,522

GO aspect counts:
Aspect
F    402622
C    255987
P    179752
Name: count, dtype: int64

Most common evidence codes:
Evidence_Code
IPI    288675
IEA    159296
IDA    108710
TAS    105115
IBA     65277
ISS     34393
IMP     26158
NAS     18449
EXP     15188
HDA      9260
Name: count, dtype: int64
